In [13]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('ggplot')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (12, 8),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import datetime
import pytz
NYC_tz = pytz.timezone("America/New_York") 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [27]:
import sys
sys.path.append("../../")

import rateslib as rl
import QuantLib as ql

from Query.IRSwaps.IRSwapQuery import IRSwapQuery 
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from Query.IRSwaps.IRSwapValue import IRSwapValue

In [28]:
curve_mdp = IRSwapsMDP(source="BARCHART_STIRF-RL")

In [75]:
# ts = NYC_tz.localize(datetime.datetime(2026, 5, 1, 17, 00))
ts = "live"
curve = "USD-SOFR-1D-Q12STIRT"

curve_handle = curve_mdp.get_pricer(request=dict(curve_name=curve, timestamp=ts))
curve_handle

RLIRSwapCurve(_rl_curve_id='USD-SOFR-1D', _rl_curve_handle=<rl.Curve:USD-SOFR-1D at 0x1629d8fdf30>, _meta_data={'timestamp': datetime.datetime(2026, 5, 4, 14, 27, 0, 895985, tzinfo=<DstTzInfo 'America/New_York' EDT-1 day, 20:00:00 DST>), 'id': 'BARCHART_STIRF-RL-USD-SOFR-1D-Q12STIRT-2026-05-04 14:27:00.895985-04:00', 'requested_curve_name': 'USD-SOFR-1D-Q12STIRT', 'curve_name': 'USD-SOFR-1D-Q12STIRT'})

In [76]:
risk_model_tenors = [
	# "fomc_1",
	# "fomc_2",
	# "fomc_3",
	# "fomc_4",
	# "fomc_5",
	# "fomc_6",
	# "fomc_7",
	
	"IMM_1xIMM_2",
    "IMM_2xIMM_3",
    "IMM_3xIMM_4",
    "IMM_4xIMM_5",
    "IMM_5xIMM_6",
    "IMM_6xIMM_7",
    "IMM_7xIMM_8",
    "IMM_8xIMM_9",
    "IMM_9xIMM_10",
    "IMM_10xIMM_11",
    "IMM_11xIMM_12",
    "IMM_12xIMM_13",
    
	# "IMM_13xIMM_14",
    # "IMM_14xIMM_15",
    # "IMM_15xIMM_16",
    # "IMM_16xIMM_17",

	# "IMM_17xIMM_18",
    # "IMM_18xIMM_19",
    # "IMM_19xIMM_20",
    # "IMM_20xIMM_21",
]
rl_risk_instruments = {}
for t in risk_model_tenors:
    outright_query = IRSwapQuery(curve=curve, tenor=t).resolve_query(ts, pricer_or_curve=curve_handle)
    outright_pkg, _ = outright_query.resolve_package(pricer_or_curve=curve_handle)
    rl_risk_instruments[t] = outright_pkg[0]

rl_risk_solver = rl.Solver(
    curves=[curve_handle.handle()],
    instruments=rl_risk_instruments.values(),
    instrument_labels=rl_risk_instruments.keys(),
    s=[r.rate().real for r in rl_risk_instruments.values()],
    id=curve_handle.id(),
    func_tol=1e-8,
    conv_tol=1e-10,
)

SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0011s


In [77]:
risk = 100_000

tenor = "IMM_U26xIMM_Z26/IMM_H27xIMM_M27/IMM_U27xIMM_Z27"
# tenor = "IMM_Z26xIMM_H27"
query = IRSwapQuery(curve=curve, tenor=tenor, structure_kwargs={"bpv": risk}).resolve_query(
    ts, pricer_or_curve=curve_handle
)
pkg, rws = query.resolve_package(pricer_or_curve=curve_handle)

vmap = query.build_value_map(pricer_or_curve=curve_handle, package=pkg, risk_weights=rws)
for v in [IRSwapValue.RATE, IRSwapValue.NPV, IRSwapValue.NOTIONAL, IRSwapValue.PV01]:
    print(v.name, vmap.apply(value=v))

print(curve_handle.effective_date(pkg[0]))
print(curve_handle.maturity_date(pkg[0]))
# print(vmap.apply(value=IRSwapValue.CARRY_BPS_RUNNING, **{"horizon": "3m"}))
print(vmap.apply(value=IRSwapValue.ROLL_BPS_RUNNING, **{"horizon": "2m"}))

display(rl.Portfolio(pkg).delta(solver=rl_risk_solver).style.format("{:_.0f}"))

RATE 21.847081465403566
NPV 0.0
NOTIONAL -571764.0752556324
PV01 7.275957614183426e-12
2026-09-16 00:00:00
2026-12-16 00:00:00
1.657799589392939


In [47]:
# curve_handle.handle().plot("1y")

queries = [
    # IRSwapQuery(curve=curve, tenor="IMM_H26xIMM_M26", structure_kwargs={"bpv": risk}).resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_M26xIMM_U26").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_U26xIMM_Z26").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_Z26xIMM_H27").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_H27xIMM_M27").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_M27xIMM_U27").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_U27xIMM_Z27").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_Z27xIMM_H28").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_H28xIMM_M28").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_M28xIMM_U28").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_U28xIMM_Z28").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_Z28xIMM_H29").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_H29xIMM_M29").resolve_query(ts, pricer_or_curve=curve_handle),
    
	IRSwapQuery(curve=curve, tenor="IMM_M29xIMM_U29").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_U29xIMM_Z29").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_Z29xIMM_H30").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_H30xIMM_M30").resolve_query(ts, pricer_or_curve=curve_handle),
	
	IRSwapQuery(curve=curve, tenor="IMM_M30xIMM_U30").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_U30xIMM_Z30").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_H31xIMM_M31").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="IMM_M31xIMM_U31").resolve_query(ts, pricer_or_curve=curve_handle),
]

rl_spots = []
for q in queries:
    pkg, _ = q.resolve_package(pricer_or_curve=curve_handle)
    print(f"{q.col_name()}: {pkg[0].rate().real}")


    rl_spots.append(pkg[0])

x = [s.__dict__["kwargs"]["effective"] for s in rl_spots]
y = [s.rate().real for s in rl_spots]

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=y, mode="lines", name="1D"))
tick_vals = [pd.Timestamp(t).tz_localize(None) for t in x]
# tick_text = [pd.Timestamp(t).strftime("%Y-%m-%d") for t in tick_vals]
tick_text = [q.tenor for q in queries]
fig.update_layout(
    title=f"{curve_handle.meta()["id"]} | {curve_handle.meta()["timestamp"]}",
    template="plotly_dark",
    margin=dict(l=40, r=20, t=60, b=80),
    xaxis=dict(tickmode="array", tickvals=tick_vals, ticktext=tick_text, tickangle=45, showgrid=True),
    height=550,
    yaxis=dict(showgrid=True),
)
fig.update_xaxes(
    showspikes=True,
    spikecolor="white",
    spikesnap="cursor",
    spikemode="across",
    showgrid=True,
)
fig.update_yaxes(
    showspikes=True,
    spikecolor="white",
    spikesnap="cursor",
    spikethickness=0.5,
    showgrid=True,
)
fig.show()

USD-SOFR-1D-Q20STIRT IMM_M26xIMM_U26 OUTRIGHT RATE: 3.654857507035349
USD-SOFR-1D-Q20STIRT IMM_U26xIMM_Z26 OUTRIGHT RATE: 3.7190866108468907
USD-SOFR-1D-Q20STIRT IMM_Z26xIMM_H27 OUTRIGHT RATE: 3.7976880004442757
USD-SOFR-1D-Q20STIRT IMM_H27xIMM_M27 OUTRIGHT RATE: 3.8555799503674013
USD-SOFR-1D-Q20STIRT IMM_M27xIMM_U27 OUTRIGHT RATE: 3.8472108252051926
USD-SOFR-1D-Q20STIRT IMM_U27xIMM_Z27 OUTRIGHT RATE: 3.7787233407111143
USD-SOFR-1D-Q20STIRT IMM_Z27xIMM_H28 OUTRIGHT RATE: 3.702660417913832
USD-SOFR-1D-Q20STIRT IMM_H28xIMM_M28 OUTRIGHT RATE: 3.6641632534100137
USD-SOFR-1D-Q20STIRT IMM_M28xIMM_U28 OUTRIGHT RATE: 3.6554488389200825
USD-SOFR-1D-Q20STIRT IMM_U28xIMM_Z28 OUTRIGHT RATE: 3.664121756993294
USD-SOFR-1D-Q20STIRT IMM_Z28xIMM_H29 OUTRIGHT RATE: 3.6833380404671106
USD-SOFR-1D-Q20STIRT IMM_H29xIMM_M29 OUTRIGHT RATE: 3.7084472799064176
USD-SOFR-1D-Q20STIRT IMM_M29xIMM_U29 OUTRIGHT RATE: 3.73648968219802
USD-SOFR-1D-Q20STIRT IMM_U29xIMM_Z29 OUTRIGHT RATE: 3.766085201066858
USD-SOFR-1D-

In [52]:
ts = "live"
curve = "USD-OIS-Q12xM12STIRT-SERFFX-MIX23"

curve_handle = curve_mdp.get_pricer(request=dict(curve_name=curve, timestamp=ts))
curve_handle
queries = [
    # IRSwapQuery(curve=curve, tenor="fomc_apr26").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="fomc_jun26").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="fomc_jul26").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="fomc_sep26").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="fomc_oct26").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="fomc_dec26").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="fomc_jan27").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="fomc_mar27").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="fomc_apr27").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="fomc_jun27").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="fomc_jul27").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="fomc_sep27").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="fomc_oct27").resolve_query(ts, pricer_or_curve=curve_handle),
    IRSwapQuery(curve=curve, tenor="fomc_dec27").resolve_query(ts, pricer_or_curve=curve_handle),
]

rl_spots = []
for q in queries:
    pkg, _ = q.resolve_package(pricer_or_curve=curve_handle)
    print(f"{q.col_name()}: {pkg[0].rate().real}")


fwd_tenor = "1d"
x_data_num, y_data_rate = curve_handle.handle()._plot_rates(fwd_tenor, left=rl.NoInput(0), right=rl.NoInput(0))
plot_data_dict = dict(zip(x_data_num, [y.real for y in y_data_rate]))

calendar = ql.UnitedStates(ql.UnitedStates.FederalReserve)
filtered_plot_data = {dt: rate for dt, rate in plot_data_dict.items() if calendar.isBusinessDay(ql.Date(dt.day, dt.month, dt.year))}

x_business_days = list(filtered_plot_data.keys())
y_business_rates = list(filtered_plot_data.values())
curve_nodes = curve_handle.handle().nodes._nodes.keys()

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_business_days, y=y_business_rates, mode="lines", name="1D"))

tick_vals = [pd.Timestamp(t).tz_localize(None) for t in curve_nodes]
tick_text = [pd.Timestamp(t).strftime("%Y-%m-%d") for t in tick_vals]

fig.update_layout(
    title=f"{curve_handle.meta()["id"]} | {curve_handle.meta()["timestamp"]} | 1d curve",
    template="plotly_dark",
    margin=dict(l=40, r=20, t=60, b=80),
    xaxis=dict(tickmode="array", tickvals=tick_vals, ticktext=tick_text, tickangle=45, showgrid=True),
    height=550,
    # width=1200,
    yaxis=dict(showgrid=True),
)
fig.update_xaxes(
    showspikes=True,
    spikecolor="white",
    spikesnap="cursor",
    spikemode="across",
    showgrid=True,
)
fig.update_yaxes(
    showspikes=True,
    spikecolor="white",
    spikesnap="cursor",
    spikethickness=0.5,
    showgrid=True,
)

fig.show()

USD-OIS-Q12xM12STIRT-SERFFX-MIX23 fomc_jun26 OUTRIGHT RATE: 3.6251834763355104
USD-OIS-Q12xM12STIRT-SERFFX-MIX23 fomc_jul26 OUTRIGHT RATE: 3.6360563556439747
USD-OIS-Q12xM12STIRT-SERFFX-MIX23 fomc_sep26 OUTRIGHT RATE: 3.6516970361471586
USD-OIS-Q12xM12STIRT-SERFFX-MIX23 fomc_oct26 OUTRIGHT RATE: 3.6918450151777824
USD-OIS-Q12xM12STIRT-SERFFX-MIX23 fomc_dec26 OUTRIGHT RATE: 3.737833431685074
USD-OIS-Q12xM12STIRT-SERFFX-MIX23 fomc_jan27 OUTRIGHT RATE: 3.770985068486356
USD-OIS-Q12xM12STIRT-SERFFX-MIX23 fomc_mar27 OUTRIGHT RATE: 3.7963360843684995
USD-OIS-Q12xM12STIRT-SERFFX-MIX23 fomc_apr27 OUTRIGHT RATE: 3.812689385757549
USD-OIS-Q12xM12STIRT-SERFFX-MIX23 fomc_jun27 OUTRIGHT RATE: 3.806860597130497
USD-OIS-Q12xM12STIRT-SERFFX-MIX23 fomc_jul27 OUTRIGHT RATE: 3.7831322237192184
USD-OIS-Q12xM12STIRT-SERFFX-MIX23 fomc_sep27 OUTRIGHT RATE: 3.7487303124957374
USD-OIS-Q12xM12STIRT-SERFFX-MIX23 fomc_oct27 OUTRIGHT RATE: 3.709707870812056
USD-OIS-Q12xM12STIRT-SERFFX-MIX23 fomc_dec27 OUTRIGHT RAT